# H-TRACE — Smart Manager End-to-End Demo




## Setup

Make the `src` package importable and point the data loader at the bundled dataset.

In [ ]:
# --- Setup: make the research code importable and point at the bundled dataset ---
# Works wherever this notebook sits (experiments/, repo root, etc.): it walks up
# the folder tree to find the research code (the folder containing src/).
import sys
from pathlib import Path

_here = Path.cwd().resolve()
_search = [_here, *_here.parents]
CODE_DIR = next((p for p in _search if (p/'src').is_dir() and (p/'experiments').is_dir()), _here)
sys.path.insert(0, str(CODE_DIR))

from src import config as C  # noqa: E402

# Find the bundled dataset (3_Dataset/) by searching up the tree; fall back to
# the research code's default data/raw/ location.
_ds_name = 'network_operator_KPIs_time_series_dataset'
_cands = [CODE_DIR/'data'/'raw'/'extracted'/_ds_name]
_cands += [p/'3_Dataset'/_ds_name for p in _search]
_ds = next((c for c in _cands if c.is_dir()), None)
if _ds is not None:
    C.DATA_RAW = _ds
    C.REAL_DIR = _ds/'data_real'
    C.SERIES_DIR = _ds/'data_series'
    C.INCIDENTS_FILE = _ds/'data_real_incidents.txt'
    C.REAL_INFO_FILE = _ds/'data_real_info.txt'
    C.SERIES_INFO_FILE = _ds/'data_series_info.txt'
print('research code:', CODE_DIR)
print('dataset      :', _ds)

## Imports

In [ ]:
from __future__ import annotations

import sys
import warnings

warnings.filterwarnings("ignore")

# Make output safe on every terminal (Windows consoles default to cp1252).
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

import numpy as np

from src import config as C
from src.data_loader import load_all
from src.preprocessing import build_feature_frame
from src.models.anomaly_detector import IsolationForestDetector
from src.models.child_agent import AgentObservation, ChildAgent
from src.models.safety_gate import Action, ActionType, SafetyGate
from src.models.smart_manager import SmartManager

## Helpers

A banner printer and `run_one_request` — pushes a single plain-language request through Smart Manager → Child Agent → Safety Gate → equipment.

In [ ]:
LINE = "=" * 74


def banner(title: str) -> None:
    print("\n" + LINE)
    print(title)
    print(LINE)


def run_one_request(operator_request: str, manager: SmartManager,
                    agent: ChildAgent, gate: SafetyGate,
                    series_id: str, feat_frame, values: np.ndarray) -> None:
    """Push a single plain-language request through the full pipeline."""
    print(f"\nOperator says:  \"{operator_request}\"")

    # 1) AI tier — Gemini (or keyword fallback) reads the request.
    decision = manager.classify(operator_request)
    print(f"  [1] Smart Manager (AI, non-real-time)")
    print(f"      intent     = {decision.intent}")
    print(f"      confidence = {decision.confidence:.2f}   via {decision.source}")
    if decision.reasoning:
        print(f"      reasoning  = {decision.reasoning}")

    # 2) ML tier — the child agent decides a concrete action on real data.
    #    We use the most recent slice of a real KPI series as "live telemetry".
    end_idx = len(values) - 1
    obs = AgentObservation(
        recent_values=values[end_idx - C.EPISODE_WINDOW:end_idx],
        feat_row=feat_frame.iloc[[end_idx]],
        cell_id=series_id,
        intent=decision.intent,
    )
    out = agent.step(obs)
    print(f"  [2] Child Agent (ML: Isolation Forest + load prediction)")
    print(f"      fault_detected = {out.fault_detected}   "
          f"predicted_load = {out.predicted_load:.0f}/1000")
    print(f"      -> proposes action: {out.action.type.value} on cell {series_id}")

    # 3) Rules tier — the Safety Gate screens the action.
    gate_decision = gate.check(out.action)
    verdict = "APPROVED [OK]" if gate_decision.approved else "BLOCKED [X]"
    print(f"  [3] Safety Gate (deterministic rules, NOT AI): {verdict}")
    if gate_decision.blocked:
        print(f"      reasons = {gate_decision.reasons}")
        manager.on_gate_decision(out.action.source, gate_decision)
    print(f"  [4] Equipment: "
          f"{'executes the action' if gate_decision.approved else 'receives NOTHING (unsafe action stopped)'}")

## Build the pipeline

Create the Smart Manager, load one real KPI cell as live telemetry, and build the Child Agent + Safety Gate.

In [ ]:
np.random.seed(C.RANDOM_SEED)

banner("H-TRACE — Gemini Smart Manager end-to-end demo")

# --- build the (small, fast) pipeline once -------------------------- #
manager = SmartManager()                       # follows config.GEMINI_ENABLED
print(manager._get_classifier().status() if manager._get_classifier()
      else "Gemini DISABLED — using deterministic keyword fallback")

series = load_all(sample=True)
real = next(s for s in series if s.kind == "real")   # a labelled real cell
feat = build_feature_frame(real)
detector = IsolationForestDetector().fit(feat)
agent = ChildAgent(area=real.area, detector=detector, predictor=None)
gate = SafetyGate()
print(f"Loaded real KPI cell {real.series_id} "
      f"({real.kpi_type}, {real.n} samples) for the live-telemetry demo.")

## Part A — three everyday operator requests

Energy-saving, peak-capacity, and self-healing intents, each run end to end.

In [ ]:
for req in [
    "It's the middle of the night and traffic is low, please save energy.",
    "Big festival in the area tonight - make sure we can handle the peak.",
    "We're seeing an outage on a cell, restore service as fast as you can.",
]:
    run_one_request(req, manager, agent, gate, real.series_id, feat, real.values)

## Part B — the Safety Gate is the last word

Hand the gate deliberately **unsafe** actions (the kind a buggy/hallucinating upstream component might emit) and watch the deterministic rules block every one.

In [ ]:
banner("Part B — the Safety Gate is the last word (even if the AI is wrong)")
print("We hand the Safety Gate a deliberately UNSAFE action — the kind a\n"
      "buggy or 'hallucinating' upstream component might ever produce — and\n"
      "show the deterministic gate blocks it no matter what the AI intended.")
unsafe_examples = [
    Action(ActionType.SLEEP_CELL, "demo", predicted_load=920.0),     # sleep a busy cell
    Action(ActionType.SCALE_POWER, "demo", predicted_load=500.0, power_pct=175.0),  # >100%
    Action(ActionType.WAKE_CELL, "demo", predicted_load=300.0, fault_active=True),  # optimise during fault
]
for a in unsafe_examples:
    d = gate.check(a)
    print(f"\n  proposed: {a.type.value:<12} (load={a.predicted_load:.0f}, "
          f"power={a.power_pct}, fault_active={a.fault_active})")
    print(f"  gate     : {'APPROVED [OK]' if d.approved else 'BLOCKED [X] ' + str(d.reasons)}")

## Summary

In [ ]:
banner("Summary")
print("- Gemini handled the human language (non-real-time, AI tier).")
print("- The concrete actions came from ML, never from the LLM.")
print("- The deterministic Safety Gate approved safe actions and blocked")
print("  every unsafe one, so generative AI can never drive the equipment.")
print(f"\nIntent decisions this run: {len(manager.intent_log)} "
      f"(via {', '.join(sorted({r.source for r in manager.intent_log}))})")